# Processamiento de datos Shield


#### By: Ian Moreno - A01178247



## Libraries 



In [ ]:
import os
import traceback
import pandas as pd
from pandasgui import show
from IPython.display import display



## General Variables 

In [ ]:
pd.set_option("display.max_columns", None)
target_folder = "Datos_disciplinas"
file_name = "df_final.csv"
route = os.path.join(target_folder,file_name)

### Checar Concentimiento y Atletas en total

In [40]:
def concent(df):
    si_concent = df[~df["C000 ID"].isna()]
    if not si_concent.empty:
        display(si_concent)
    else:
        print("No hay concent en el DataFrame.")

    #display(si_concent)

def count_athletes(df):
    unique_athletes = df["Athlete"].nunique()
    display(unique_athletes.value_counts())





## Aux

In [ ]:
def preguntar_sn(mensaje):
    while True:
        r = input(mensaje).strip().lower()
        if r in ("s", "si", "sí"):
            return True
        if r in ("n", "no"):
            return False
        print("Introduce s o n.")


def pedir_nombres(validos):
    while True:
        texto = input("Nombres exactos separados por comas (Enter para cancelar): ")
        nombres = [n.strip() for n in texto.split(",") if n.strip()]
        if not nombres:
            return None
        #Validar los nombres en la lista
        no_existen = [n for n in nombres if n not in validos]
        if not no_existen:
            return nombres
        print("No están en la lista:", ", ".join(no_existen))


def pedir_saltos(validos):
    while True:
        texto = input(
            "Números de los saltos a eliminar, separados por comas "
            "(Enter para cancelar): "
        )
        partes = [p.strip() for p in texto.split(",") if p.strip()]
        if not partes:
            return None
        try:
            numeros = [int(p) for p in partes]
        except ValueError:
            print("Solo se permiten números enteros.")
            continue
        invalidos = [n for n in numeros if n not in validos]
        if not invalidos:
            return numeros
        print("No pertenecen a los atletas seleccionados:", invalidos)


# Buscar todos los files 
def files_general(target_folder, word):
    file_list = []

    for file in os.listdir(target_folder):
        if file.lower().endswith(".xlsx") and word.lower() in file.lower():
            file_list.append(file)

    return file_list




## Jump Count

In [ ]:
def jump_count(df, sheet_name):

    df_tmp = df.copy()

    # El índice identifica cada salto, así que debe ser único
    if not df_tmp.index.is_unique:
        df_tmp = df_tmp.reset_index(drop=True)
    
    columnas_base = [
        "Forms ID",
        "C000 ID",
        "Athlete",
        "Date of Birth",
        "Genero",
        "Disciplina",
        "Test Type",
        "Test Date",
        "Body Weight [kg]",
        "Trial",
        "Jump Height (Imp-Mom) [cm]",
        "Concentric Impulse (Left) [N s]",
        "Concentric Impulse (Right) [N s]",
        "Concentric RFD (Left) [N/s]",
        "Concentric RFD (Right) [N/s]",
        "Eccentric Braking RFD (Left) [N/s]",
        "Eccentric Braking RFD (Right) [N/s]",
        "RSI-modified (Imp-Mom) [m/s]",
        "Takeoff Peak Force (Right) [N]",
        "Takeoff Peak Force (Left) [N]",
        "Concentric Time to Peak Force (Left) [ms]",
        "Concentric Time to Peak Force (Right) [ms]",
        "Force at Peak Power (Left) [N]",
        "Force at Peak Power (Right) [N]",
        "Peak Landing Force (Left) [N]",
        "Peak Landing Force (Right) [N]",
        "Landing RFD (Right) [N/s]",
        "Landing RFD (Left) [N/s]",
        "Concentric Peak Force (Left) [N]",
        "Concentric Peak Force (Right) [N]",
        "Eccentric Peak Force (Left) [N]",
        "Eccentric Peak Force (Right) [N]",
    ]

    # Se filtran las columnas que existen en el DataFrame
    cols_mostrar = [c for c in columnas_base if c in df_tmp.columns]

    while True:
        # Se cuentan los intentos de cada atleta
        conteo = df_tmp["Athlete"].value_counts().reset_index()
        conteo.columns = ["Athlete", "Trials"]
        reprobados = conteo[conteo["Trials"] != 3]
    
        if reprobados.empty:
            print(f"No hay atletas con un número de intentos distinto de 3 "
                  f"en la sesión '{sheet_name}'.")
            return df_tmp

        #Aqui se filtran los atletas que no tienen 3 intentos y se muestran sus saltos
        df_reprobados = df_tmp.loc[
            df_tmp["Athlete"].isin(reprobados["Athlete"]), cols_mostrar
        ].sort_values("Athlete")

        print(f"\n=== Sesión '{sheet_name}' ===")
        print("Atletas con intentos distintos de 3:")
        display(reprobados)
        print("Sus saltos:")
        display(df_reprobados)

        while True:
            accion = input(
                "\n¿Qué quieres hacer?\n"
                "  [1] Eliminar a todos los atletas de la lista\n"
                "  [2] Revisar atletas específicos\n"
                "  [3] Terminar sin eliminar más\n"
                "Opción: "
            ).strip()
            if accion in ("1", "2", "3"):
                break
            print("Introduce 1, 2 o 3. No hay atletas con un número de intentos distinto de 3 en la sesión 'Sesion1'")

        if accion == "1":
            df_tmp = df_tmp.loc[
                ~df_tmp["Athlete"].isin(reprobados["Athlete"])
            ].copy()
            print("Se eliminaron los atletas de la lista.")
            return df_tmp

        if accion == "3":
            return df_tmp

        nombres = pedir_nombres(set(reprobados["Athlete"]))
        if nombres is None:
            continue

        saltos = df_tmp.loc[df_tmp["Athlete"].isin(nombres), cols_mostrar]
        print("\nSaltos de los atletas seleccionados:")
        display(saltos)

        numeros = pedir_saltos(set(saltos.index))
        if numeros is None:
            continue

        print("\nSe eliminarán estos saltos:")
        display(df_tmp.loc[numeros, cols_mostrar])

        if preguntar_sn(f"¿Eliminar {len(numeros)} salto(s)? (s/n): "):
            df_tmp = df_tmp.drop(index=numeros)
            print("Saltos eliminados.")
        else:
            print("No se eliminó nada.")

        if not preguntar_sn("¿Seguir revisando? (s/n): "):
            return df_tmp

## Main

#### Mostrar Cuantos documentos hay, los deportes y sus ramas

In [ ]:
# Contar el número de documentos en la carpeta
try:
    file_count = sum(1 for entry in os.scandir(target_folder) if entry.is_file())
    print(f"Total de documentos: {file_count - 1}")

except FileNotFoundError:
    print(f"Error: no se encontró la carpeta '{target_folder}'.")


# Contar y nombrar la cantidad de deportes
sports = []
files = []

print("Lista de deportes:")

for file in os.listdir(target_folder):
    if file.endswith(".xlsx"):
        files.append(file)

        name = file.rsplit(".", 1)[0]
        sports.append(name)

        print(f"{len(sports)}: {name}")


#### Mostrar cuantas hojas hay en cada excel

In [ ]:
print(files)

In [ ]:
# Contar cuantos documentos son  
count_number(target_folder)

#Mostrar  deportes y ramas
sports, files = ramas_d()

#Lee df concat
df = pd.read_csv(route)

# Mostrar cuantas hojas hay para todos los files de exceles



while True:
    sport_input = input("Ingrese el numero del deporte a procesar: ")

    if not (sport_input.isdigit() and 1 <= int(sport_input) <= len(files)):
        print("Deporte no encontrado.")
        continue

    else:
            break


## Show final df

In [ ]:
#show(**sesiones_finales)                                # todas las sesiones en PandasGUI
#todo = pd.concat(sesiones_finales, names=["Sesion"])    # un solo DataFrame para analizar